In [1]:
from abc import ABC, abstractmethod
import pandas as pd
import numpy as np
import yfinance as yf
import plotly.graph_objects as go
import pickle
import os
from pathlib import Path  

# Mapping da ticker a nomi delle cryptocurrency per esteso
CRYPTO_NAMES = {
    'BTC-USD': 'Bitcoin',
    'ETH-USD': 'Ethereum',
    'XRP-USD': 'XRP',
    'SOL-USD': 'Solana',
    'DOGE-USD': 'Dogecoin',
    'ADA-USD': 'Cardano',
    'LINK-USD': 'Chainlink',
    'AVAX-USD': 'Avalanche',
    'DOT-USD': 'Polkadot',
    'UNI7083-USD': 'Uniswap',
    'SKY33038-USD': 'Sky',
    'NEAR-USD': 'Near',
    'ATOM-USD': 'Cosmos',
    'POL28321-USD': 'Polygon',
    'ALGO-USD': 'Algorand',
    'APT21794-USD': 'Aptos',
    'ARB11841-USD': 'Arbitrum',
    'STX4847-USD': 'Stacks',
    'INJ-USD': 'Injective',
    'TIA-USD': 'Celestia',
    'GRT6719-USD': 'The Graph',
    'OP-USD': 'Optimism',
    'SUI20947-USD': 'Sui',
    'XLM-USD': 'Stellar Lumens',
    'XPL-USD': 'XPL',
    'ONDO-USD': 'Ondo',
    'HBAR-USD': 'Hedera',
    'FIL-USD': 'Filecoin',
    'AAVE-USD': 'Aave',
    'NIGHT39064-USD': 'Nyx',
    'ETC-USD': 'Ethereum Classic',
    'LTC-USD': 'Litecoin',
}

/Users/Lorenzo/index/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
# ==========================================
# 1. CLASSE GENITORE (BASE)
# ==========================================
class BaseCryptoStrategy(ABC):
    """
    Classe genitore che gestisce il recupero dati, le statistiche e i grafici interattivi.
    Le classi figlie devono implementare solo la logica di 'run_strategy'.
    """
    
    def __init__(self, initial_value: float = 10000.0):
        self.initial_value = initial_value
        self.data_dict = {}
        self.assets = []
        self.index_df = None
        self.benchmark_df = None  
        self.benchmark_ticker = 'BTC-USD' 
        self.rebalance_dates = [] # <-- Nuovo attributo per tracciare le date di ribilanciamento
        self.cache_dir = Path('./data_cache')  # Directory per cachare i dati
        self.cache_dir.mkdir(exist_ok=True)  # Crea la directory se non esiste

    def _get_cache_path(self, ticker_symbol: str, timeframe: str) -> Path:
        """Restituisce il percorso del file cache per un ticker."""
        cache_file = f"{ticker_symbol}_{timeframe}.pkl"
        return self.cache_dir / cache_file
    
    def _load_cache(self, ticker_symbol: str, timeframe: str):
        """Carica i dati dalla cache se disponibili."""
        cache_path = self._get_cache_path(ticker_symbol, timeframe)
        if cache_path.exists():
            try:
                with open(cache_path, 'rb') as f:
                    data = pickle.load(f)
                    return data
            except Exception as e:
                print(f"  Errore nel caricamento della cache per {ticker_symbol}: {e}")
        return None
    
    def _save_cache(self, ticker_symbol: str, timeframe: str, data):
        """Salva i dati nella cache."""
        cache_path = self._get_cache_path(ticker_symbol, timeframe)
        try:
            with open(cache_path, 'wb') as f:
                pickle.dump(data, f)
        except Exception as e:
            print(f"  Errore nel salvataggio della cache per {ticker_symbol}: {e}")

    def import_data(self, assets: list, timeframe: str = "1y", use_cache: bool = True, refresh_cache: bool = False):
        """Scarica e prepara i dati base (Prezzo, Volume, Circulating Supply) per la lista di asset.
        
        Args:
            assets: Lista di ticker da scaricare
            timeframe: Periodo storico (es. '1y', 'max')
            use_cache: Se True, usa i dati salvati in cache se disponibili
            refresh_cache: Se True, scarica i nuovi dati anche se la cache esiste e la aggiorna
        """
        print(f"Inizio download dati per il periodo '{timeframe}'...")
        if use_cache and not refresh_cache:
            print("(usando cache se disponibile)")
        self.assets = []
        
        for ticker_symbol in assets:
            try:
                circulating_supply = None
                # Prova a caricare dalla cache
                if use_cache and not refresh_cache:
                    cached_data = self._load_cache(ticker_symbol, timeframe)
                    if cached_data is not None:
                        try:
                            # Tenta a unpackare come tupla (nuovo formato)
                            df_raw, circulating_supply = cached_data
                            print(f"  {ticker_symbol}: caricato dalla cache")
                        except (ValueError, TypeError):
                            # Se non è una tupla, potrebbe essere il vecchio formato (solo df_raw)
                            # In questo caso, scarica di nuovo per safety
                            ticker = yf.Ticker(ticker_symbol)
                            df_raw = ticker.history(period=timeframe)
                            circulating_supply = ticker.info.get('circulatingSupply')
                            self._save_cache(ticker_symbol, timeframe, (df_raw, circulating_supply))
                            print(f"  {ticker_symbol}: scaricato (cache non compatibile) e salvato in cache")
                    else:
                        ticker = yf.Ticker(ticker_symbol)
                        df_raw = ticker.history(period=timeframe)
                        circulating_supply = ticker.info.get('circulatingSupply')
                        self._save_cache(ticker_symbol, timeframe, (df_raw, circulating_supply))
                        print(f"  {ticker_symbol}: scaricato e salvato in cache")
                else:
                    ticker = yf.Ticker(ticker_symbol)
                    df_raw = ticker.history(period=timeframe)
                    circulating_supply = ticker.info.get('circulatingSupply')
                    if refresh_cache:
                        self._save_cache(ticker_symbol, timeframe, (df_raw, circulating_supply))
                        print(f"  {ticker_symbol}: scaricato e cache aggiornata")
                    else:
                        print(f"  {ticker_symbol}: scaricato")
                
                if df_raw.empty:
                    print(f"Nessun dato per {ticker_symbol}. Salto.")
                    continue
                    
                if not circulating_supply:
                    print(f"Circulating Supply mancante per {ticker_symbol}. Salto.")
                    continue
                    
                df_final = df_raw[['Close', 'Volume']].copy()
                df_final['Circulating Supply'] = circulating_supply
                df_final.columns = ['Prezzo di Chiusura', 'Volume 24h', 'Circulating Supply']
                
                self.data_dict[ticker_symbol] = df_final
                self.assets.append(ticker_symbol)
                
            except Exception as e:
                print(f"Errore durante il recupero di {ticker_symbol}: {e}")

        # --- GESTIONE DEL BENCHMARK ---
        if self.benchmark_ticker in self.data_dict:
            self.benchmark_df = self.data_dict[self.benchmark_ticker]
        else:
            print(f"\n{self.benchmark_ticker} non presente nel dizionario. Avvio download in un df separato...")
            try:
                circulating_supply_bench = None
                # Prova a caricare dalla cache
                if use_cache and not refresh_cache:
                    cached_bench = self._load_cache(self.benchmark_ticker, timeframe)
                    if cached_bench is not None:
                        df_bench_raw, circulating_supply_bench = cached_bench
                        print(f"Benchmark {self.benchmark_ticker}: caricato dalla cache\n")
                    else:
                        ticker_bench = yf.Ticker(self.benchmark_ticker)
                        df_bench_raw = ticker_bench.history(period=timeframe)
                        circulating_supply_bench = ticker_bench.info.get('circulatingSupply')
                        self._save_cache(self.benchmark_ticker, timeframe, (df_bench_raw, circulating_supply_bench))
                        print(f"Benchmark {self.benchmark_ticker}: scaricato e salvato in cache\n")
                else:
                    ticker_bench = yf.Ticker(self.benchmark_ticker)
                    df_bench_raw = ticker_bench.history(period=timeframe)
                    circulating_supply_bench = ticker_bench.info.get('circulatingSupply')
                    if refresh_cache:
                        self._save_cache(self.benchmark_ticker, timeframe, (df_bench_raw, circulating_supply_bench))
                
                if not df_bench_raw.empty:
                    df_bench_final = df_bench_raw[['Close']].copy()
                    df_bench_final.columns = ['Prezzo di Chiusura']
                    self.benchmark_df = df_bench_final
                    if not (use_cache and not refresh_cache and self._load_cache(self.benchmark_ticker, timeframe) is not None):
                        print(f"Benchmark {self.benchmark_ticker} completato con successo.\n")
                else:
                    print(f"Nessun dato trovato per il benchmark {self.benchmark_ticker}.\n")
            except Exception as e:
                print(f"Errore durante il download del benchmark {self.benchmark_ticker}: {e}\n")

    @abstractmethod
    def run_strategy(self, **kwargs):
        """Metodo astratto: ogni classe figlia deve definire la propria logica di backtest."""
        pass

    def plot_results(self):
        """Grafica l'andamento della strategia confrontato con il benchmark (Interattivo)."""
        if self.index_df is None:
            raise ValueError("Strategia non calcolata. Esegui run_strategy() prima di plottare.")
            
        fig = go.Figure()

        # --- PLOT STRATEGIA ---
        fig.add_trace(go.Scatter(
            x=self.index_df.index, 
            y=self.index_df['Valore Indice'],
            mode='lines',
            name='Conio MarketCap Index',
            line=dict(color='#1f77b4', width=2)
        ))
        
        # --- PLOT BENCHMARK ---
        if self.benchmark_df is not None:
            common_idx = self.index_df.index.intersection(self.benchmark_df.index)
            if not common_idx.empty:
                btc_prices = self.benchmark_df['Prezzo di Chiusura'].loc[common_idx]
                btc_normalized = (btc_prices / btc_prices.iloc[0]) * self.initial_value
                
                fig.add_trace(go.Scatter(
                    x=btc_normalized.index, 
                    y=btc_normalized,
                    mode='lines',
                    name=f'{self.benchmark_ticker} (Benchmark)',
                    line=dict(color='#cc8306', width=2)
                ))
        
        # --- LINEA CAPITALE INIZIALE ---
        fig.add_hline(
            y=self.initial_value, 
            line_dash="dash", 
            line_color="red", 
            annotation_text=f"Capitale Iniziale ({self.initial_value}$)", 
            annotation_position="bottom right"
        )

        # --- LINEE DEI RIBILANCIAMENTI ---
        # Aggiungiamo una linea verticale per ogni data in cui è avvenuto un ribilanciamento
        if self.rebalance_dates:
            for date in self.rebalance_dates:
                fig.add_vline(
                    x=date, 
                    line_width=1, 
                    line_dash="dot", 
                    line_color="green",
                    opacity=0.6
                )
            # Aggiungo una riga invisibile solo per far comparire la legenda dei ribilanciamenti
            fig.add_trace(go.Scatter(
                x=[None], y=[None], mode='lines',
                line=dict(color='green', dash='dot'),
                name='Ribilanciamento'
            ))

        # --- LAYOUT INTERATTIVO ---
        fig.update_layout(
            title="Conio MarketCap Index vs Benchmark",
            xaxis_title="Data",
            yaxis_title="Valore del Conio Index",
            hovermode="x unified", # Mostra i dati di tutte le curve passando il mouse su un punto
            template="plotly_white",
            legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01),
            yaxis_type="log"  # Scala logaritmica
        )
        
        fig.show()

    def calculate_stats(self):
        """Calcola e restituisce le statistiche della strategia rispetto al benchmark."""
        if self.index_df is None:
            raise ValueError("Strategia non calcolata. Esegui run_strategy().")
        
        strat_returns = self.index_df['Valore Indice'].pct_change().dropna()
        stats = {'Strategia': self._compute_metrics(strat_returns)}
        
        if self.benchmark_df is not None:
            common_idx = self.index_df.index.intersection(self.benchmark_df.index)
            if not common_idx.empty:
                btc_prices = self.benchmark_df['Prezzo di Chiusura'].loc[common_idx]
                btc_returns = btc_prices.pct_change().dropna()
                stats[self.benchmark_ticker] = self._compute_metrics(btc_returns)
            
        return pd.DataFrame(stats).round(4)

    def _compute_metrics(self, returns: pd.Series) -> dict:
        """Helper interno per il calcolo delle metriche finanziarie su 365 giorni."""
        trading_days = 365
        ann_return = returns.mean() * trading_days
        ann_volatility = returns.std() * np.sqrt(trading_days)
        sharpe_ratio = ann_return / ann_volatility if ann_volatility != 0 else 0
        
        cumulative_returns = (1 + returns).cumprod()
        peak = cumulative_returns.cummax()
        max_drawdown = ((cumulative_returns - peak) / peak).min()
        
        return {
            'Ritorno Annualizzato (%)': ann_return * 100,
            'Volatilità Annualizzata (%)': ann_volatility * 100,
            'Sharpe Ratio': sharpe_ratio,
            'Max Drawdown (%)': max_drawdown * 100
        }

In [3]:
# ==========================================
# 2. CLASSE FIGLIA (MARKET CAP INDEX STRATEGY)
# ==========================================
class MarketCapIndexStrategy(BaseCryptoStrategy):
    """
    Strategia che crea un indice dinamico ponderato per market cap.
    Le crypto vengono aggiunte all'indice quando raggiungono il minimo market cap (default 100M USD).
    Vengono rimosse quando scendono sotto la soglia.
    Il portafoglio è sempre ponderato per market cap tra i crypto elegibili.
    """
    
    def __init__(self, initial_value: float = 10000.0):
        super().__init__(initial_value)
        self.composition_history = {}  # Traccia la composizione nel tempo
        self.composition_dates = []     # Date di ribilanciamento
    
    def run_strategy(self, rebalance_freq='ME', min_marketcap=100_000_000, **kwargs):
        """
        rebalance_freq: Frequenza di ribilanciamento. 'ME' = fine mese, 'QE' = fine trimestre, 'YE' = fine anno.
        min_marketcap: Market cap minimo (in USD) per essere incluso nell'indice. Default 100M.
        """
        if not self.data_dict:
            raise ValueError("Nessun dato disponibile. Esegui import_data() per primo.")
            
        print(f"\nAvvio backtest: Market Cap Index con soglia minima ${min_marketcap/1e6:.0f}M (frequenza ribilanciamento: {rebalance_freq})...")
        
        df_prices = pd.DataFrame()
        df_mktcap = pd.DataFrame()
        
        # 1. Allineamento dati e calcolo della Market Cap
        for ticker in self.assets:
            df = self.data_dict[ticker].copy()
            df['MarketCap'] = df['Prezzo di Chiusura'] * df['Circulating Supply']
            
            df.index = pd.to_datetime(df.index)
            df_prices[ticker] = df['Prezzo di Chiusura']
            df_mktcap[ticker] = df['MarketCap']
            
        df_prices.ffill(inplace=True)
        df_mktcap.ffill(inplace=True)
        df_prices.dropna(how='all', inplace=True)
        
        dates = df_prices.index
        first_date = dates[0]
        
        # 2. Identificazione dei giorni di ribilanciamento
        rebalance_dates = df_prices.resample(rebalance_freq).last().index
        rebalance_dates = [d for d in rebalance_dates if d in dates]
        
        portfolio_value = self.initial_value
        shares = {}
        current_index_cryptos = []
        index_values = []
        prev_index_cryptos = []
        
        # 3. Loop giornaliero: simulazione day-by-day
        for current_date in dates:
            
            # --- A. AGGIORNAMENTO VALORE PORTAFOGLIO ---
            if shares:
                portfolio_value = sum(shares[ticker] * df_prices.loc[current_date, ticker] for ticker in current_index_cryptos if ticker in shares)
            
            index_values.append(portfolio_value)
            
            # --- B. CONTROLLO RIBILANCIAMENTO ---
            if current_date == first_date or current_date in rebalance_dates:
                
                # Prende la Mkt Cap di TUTTE le crypto in questa specifica data
                daily_mktcap = df_mktcap.loc[current_date].dropna()
                
                # Filtra i crypto che raggiungono il minimo market cap
                eligible_cryptos = daily_mktcap[daily_mktcap >= min_marketcap]
                current_index_cryptos = eligible_cryptos.index.tolist()
                
                total_mktcap = eligible_cryptos.sum()
                if total_mktcap == 0:
                    continue
                
                # Calcola i pesi basati su market cap
                weights = (eligible_cryptos / total_mktcap).to_dict()
                
                # Traccia composizione
                self.composition_history[current_date] = weights.copy()
                self.composition_dates.append(current_date)
                
                # Stampa quando nuovi crypto entrano/escono
                new_entries = set(current_index_cryptos) - set(prev_index_cryptos)
                removed = set(prev_index_cryptos) - set(current_index_cryptos)
                
                if new_entries and current_date != first_date:
                    print(f"  {current_date.strftime('%Y-%m-%d')}: +{', '.join(sorted(new_entries))}")
                if removed:
                    print(f"  {current_date.strftime('%Y-%m-%d')}: -{', '.join(sorted(removed))}")
                
                prev_index_cryptos = current_index_cryptos.copy()
                
                # Ricalcola le quote per tutti i crypto nell'indice
                shares = {}
                for ticker in current_index_cryptos:
                    allocated_capital = portfolio_value * weights[ticker]
                    price = df_prices.loc[current_date, ticker]
                    shares[ticker] = allocated_capital / price if price > 0 else 0
                    
        # 4. Salvataggio della serie storica
        self.index_df = pd.DataFrame({'Date': dates, 'Valore Indice': index_values})
        self.index_df.set_index('Date', inplace=True)
        
        print(f"\nBacktest completato. Composizione FINALE dell'indice al {dates[-1].strftime('%Y-%m-%d')}:")
        final_mktcap = df_mktcap.loc[dates[-1]]
        for ticker in current_index_cryptos:
            if ticker in weights:
                print(f"- {ticker}: {weights[ticker]:.2%} (Mkt Cap: ${final_mktcap[ticker]:,.0f})")
    
    def plot_composition(self):
        """Visualizza come cambia la composizione dell'indice nel tempo."""
        if not self.composition_history:
            raise ValueError("Composizione non tracciata. Esegui run_strategy() prima.")
        
        # Crea un DataFrame con la composizione nel tempo
        composition_df = pd.DataFrame(self.composition_history).T
        composition_df.index = pd.to_datetime(composition_df.index)
        composition_df = composition_df.fillna(0)
        
        # Ordina i ticker per frequenza di apparizione (i più frequenti in alto)
        ticker_freq = (composition_df > 0).sum().sort_values(ascending=False)
        composition_df = composition_df[ticker_freq.index]
        
        # Crea un grafico stacked area
        fig = go.Figure()
        
        for ticker in composition_df.columns:
            crypto_name = CRYPTO_NAMES.get(ticker, ticker)  # Usa il nome per esteso o il ticker se non trovato
            fig.add_trace(go.Scatter(
                x=composition_df.index,
                y=composition_df[ticker],
                mode='lines',
                name=crypto_name,
                stackgroup='one',
                fillcolor=None,
                hovertemplate='<b>%{fullData.name}</b><br>%{x|%Y-%m-%d}<br>% del portafoglio: %{y:.1%}<extra></extra>'
            ))
        
        fig.update_layout(
            title="Evoluzione della Composizione dell'Indice nel Tempo",
            xaxis_title="Data",
            yaxis_title="Peso nel Portafoglio (%)",
            hovermode="x unified",
            template="plotly_white",
            yaxis=dict(tickformat=".0%"),
            height=600,
            showlegend=False
        )
        
        fig.show()

In [ ]:
if __name__ == "__main__":
    # 1. Define the list of assets to include in the portfolio
    # It is recommended to include "BTC-USD" so it is used as benchmark in charts and statistics

    Conio_coin = [
        "BTC-USD", "ETH-USD", "XRP-USD", "SOL-USD", "DOGE-USD", 
        "ADA-USD", "LINK-USD", "AVAX-USD", "DOT-USD", 
        "UNI7083-USD", "SKY33038-USD", "NEAR-USD", "ATOM-USD", 
        "POL28321-USD", "ALGO-USD", "APT21794-USD", "ARB11841-USD", "STX4847-USD", 
        "INJ-USD", "TIA-USD", "GRT6719-USD", "OP-USD", "SUI20947-USD", 
        "XLM-USD", "XPL-USD", "ONDO-USD", "HBAR-USD", 
        "FIL-USD", "AAVE-USD", "NIGHT39064-USD", "ETC-USD", "LTC-USD"
    ]
    
    # 2. Initialize the strategy with starting capital
    strategy = MarketCapIndexStrategy(initial_value=10000.0)
    
    # 3. Import data (e.g., full historical data)
    # use_cache=True (default): carica dati dalla cache se disponibili, altrimenti scarica e salva
    # refresh_cache=False (default): usa la cache esistente
    # Se vuoi forzare il refresh dei dati, usa: refresh_cache=True
    strategy.import_data(assets=Conio_coin, timeframe="max", use_cache=True, refresh_cache=False)
    
    # Set initial value to initial BTC price
    if 'BTC-USD' in strategy.data_dict:
        initial_btc_price = strategy.data_dict['BTC-USD']['Prezzo di Chiusura'].iloc[0]
        strategy.initial_value = initial_btc_price
        print(f"\nInitial portfolio value set to initial BTC price: ${initial_btc_price:,.2f}")
    
    # 4. Run the backtest with dynamic market cap index
    # Wrap in try-except in case of connection or missing data errors
    try:
        strategy.run_strategy(rebalance_freq='ME', min_marketcap=1_000_000_000)  # 1B USD minimum
        
        # 5. Calculate and print statistics
        print("\n--- Performance Statistics ---")
        stats_df = strategy.calculate_stats()
        print(stats_df)
        
        # 6. Display the chart of index performance
        strategy.plot_results()
        
        # 7. Display the composition evolution
        print("\n--- Index Composition Over Time ---")
        strategy.plot_composition()
        
    except ValueError as e:
        print(f"\nError during strategy execution: {e}")

Inizio download dati per il periodo 'max'...
(usando cache se disponibile)
  BTC-USD: caricato dalla cache
  ETH-USD: caricato dalla cache
  XRP-USD: caricato dalla cache
  SOL-USD: caricato dalla cache
  DOGE-USD: caricato dalla cache
  ADA-USD: caricato dalla cache
  LINK-USD: caricato dalla cache
  AVAX-USD: caricato dalla cache
  DOT-USD: caricato dalla cache
  UNI7083-USD: caricato dalla cache
  SKY33038-USD: caricato dalla cache
  NEAR-USD: caricato dalla cache
  ATOM-USD: caricato dalla cache
  POL28321-USD: caricato dalla cache
  ALGO-USD: caricato dalla cache
  APT21794-USD: caricato dalla cache
  ARB11841-USD: caricato dalla cache
  STX4847-USD: caricato dalla cache
  INJ-USD: caricato dalla cache
  TIA-USD: caricato dalla cache
  GRT6719-USD: caricato dalla cache
  OP-USD: caricato dalla cache
  SUI20947-USD: caricato dalla cache
  XLM-USD: caricato dalla cache
  XPL-USD: caricato dalla cache
  ONDO-USD: caricato dalla cache
  HBAR-USD: caricato dalla cache
  FIL-USD: carica


--- Index Composition Over Time ---


In [5]:
# Export Performance Chart for Web
import plotly.io as pio
import json
import numpy as np

# Custom JSON encoder to handle numpy arrays and pandas objects
class NumpyEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        if isinstance(obj, np.integer):
            return int(obj)
        if isinstance(obj, np.floating):
            return float(obj)
        if hasattr(obj, 'isoformat'):  # datetime objects
            return obj.isoformat()
        return super().default(obj)

# Ricrea il grafico delle performance
fig_perf = go.Figure()

# Get initial BTC price as the reference value
if strategy.benchmark_df is not None:
    common_idx = strategy.index_df.index.intersection(strategy.benchmark_df.index)
    if not common_idx.empty:
        btc_prices = strategy.benchmark_df['Prezzo di Chiusura'].loc[common_idx]
        initial_btc_price = btc_prices.iloc[0]
        
        # Normalize strategy to start at initial BTC price
        strategy_normalized = (strategy.index_df['Valore Indice'] / strategy.index_df['Valore Indice'].iloc[0]) * initial_btc_price
        
        # Normalize BTC to start at initial price
        btc_normalized = (btc_prices / btc_prices.iloc[0]) * initial_btc_price
        
        # Plot strategia
        fig_perf.add_trace(go.Scatter(
            x=strategy_normalized.index.tolist(),
            y=strategy_normalized.tolist(),
            mode='lines',
            name='Market Cap Index',
            line=dict(color='#3b82f6', width=2)
        ))
        
        # Plot benchmark
        fig_perf.add_trace(go.Scatter(
            x=btc_normalized.index.tolist(),
            y=btc_normalized.tolist(),
            mode='lines',
            name='Bitcoin (Benchmark)',
            line=dict(color='#f59e0b', width=2)
        ))
else:
    # Fallback: plot without normalization
    fig_perf.add_trace(go.Scatter(
        x=strategy.index_df.index.tolist(),
        y=strategy.index_df['Valore Indice'].tolist(),
        mode='lines',
        name='Market Cap Index',
        line=dict(color='#3b82f6', width=2)
    ))

fig_perf.update_layout(
    title="Market Cap Index Strategy Performance",
    xaxis_title="Date",
    yaxis_title="Portfolio Value ($)",
    hovermode="x unified",
    template="plotly_dark",
    height=500,
    margin=dict(l=50, r=50, t=80, b=50),
    plot_bgcolor='rgba(15, 23, 42, 0.5)',
    paper_bgcolor='rgba(0, 0, 0, 0)',
    font=dict(family='Inter, sans-serif', color='#e2e8f0'),
    legend=dict(x=0.5, y=1, xanchor='center', yanchor='top')
)

# Export as JSON with custom encoder
perf_json = fig_perf.to_dict()

with open('strategy_performance.json', 'w') as f:
    json.dump(perf_json, f, cls=NumpyEncoder)

print("✓ Performance chart exported to strategy_performance.json")


✓ Performance chart exported to strategy_performance.json


In [6]:
# Export Composition Chart for Web
import plotly.io as pio
import json
import numpy as np

# Custom JSON encoder to handle numpy arrays and pandas objects
class NumpyEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        if isinstance(obj, np.integer):
            return int(obj)
        if isinstance(obj, np.floating):
            return float(obj)
        if hasattr(obj, 'isoformat'):  # datetime objects
            return obj.isoformat()
        return super().default(obj)

# Ricrea il grafico della composizione
composition_df = pd.DataFrame(strategy.composition_history).T
composition_df.index = pd.to_datetime(composition_df.index)
composition_df = composition_df.fillna(0)

# Ordina i ticker per frequenza di apparizione
ticker_freq = (composition_df > 0).sum().sort_values(ascending=False)
composition_df = composition_df[ticker_freq.index]

fig_comp = go.Figure()

for ticker in composition_df.columns:
    crypto_name = CRYPTO_NAMES.get(ticker, ticker)
    fig_comp.add_trace(go.Scatter(
        x=composition_df.index.tolist(),
        y=composition_df[ticker].tolist(),
        mode='lines',
        name=crypto_name,
        stackgroup='one',
        fillcolor=None,
        hovertemplate='<b>%{fullData.name}</b><br>%{x|%Y-%m-%d}<br>% del portafoglio: %{y:.1%}<extra></extra>'
    ))

fig_comp.update_layout(
    title="Portfolio Composition Evolution",
    xaxis_title="Date",
    yaxis_title="Allocation (%)",
    hovermode="x unified",
    template="plotly_dark",
    yaxis=dict(tickformat=".0%"),
    height=500,
    margin=dict(l=50, r=50, t=80, b=50),
    plot_bgcolor='rgba(15, 23, 42, 0.5)',
    paper_bgcolor='rgba(0, 0, 0, 0)',
    font=dict(family='Inter, sans-serif', color='#e2e8f0'),
    showlegend=False
)

# Export as JSON with custom encoder
comp_json = fig_comp.to_dict()

with open('strategy_composition.json', 'w') as f:
    json.dump(comp_json, f, cls=NumpyEncoder)

print("✓ Composition chart exported to strategy_composition.json")


✓ Composition chart exported to strategy_composition.json
